# Luma GE workflow for default options

This notebook shows the Luma GE workflow for this user journey:
1. Module 2: 
    - User chooses RESTORE+ classification scheme
    - User selects a subset of the RESTORE+ classification scheme
2. Module 3: The system retrieve the RESTORE+ training dataset.
2. Module 6:
    - Eventhough the training dataset was retrieved on Module 3, in practice, Luma-GE does not run the usual classification algorithm (`classification.hard_classification()`). Instead, the system calls the `classification.classify_from_prebuilt()` and clip the wall to wall map according to the user's AOI. The wall to wall default maps are available from 2013 to 2025.
    - The prebuilt map is reclassified according to the user's class of interest
    - Model accuracy is not computed, instead, use the pre-determined model accuracy inside the function 
    - Training data quality is not shown in this pathway

# Setup

In [ ]:
!python -m pip install .. --quiet

In [ ]:
import ee 
import luma_ge

service_account_path = '../auth/ee-epstm2024.json'
luma_ge.initialize_with_service_account(service_account_path)

#Check authentication status
status = luma_ge.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")

# Module 1

## Upload AOI

In [ ]:
import geemap

aoi = geemap.shp_to_ee('../data/mockup/new_aoi.shp') #change directory

## Satellite imagery retrieval

In [ ]:
from luma_ge.data_acquisition import Reflectance_Data, Reflectance_Stats, final_Image

#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2025-01-01'
end = '2025-12-31'
#get the image collection and corresponding statistics
landsat_data, meta = optical_reflectance.get_optical_data(aoi, start, end, optical_data='L9_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)
#create mosaic between image collection, and clip based on AOI
mosaic_landsat = composite.get_quality_mosaic(landsat_data, aoi, quality_band= 'NDVI', calculate_coverage=False) #REPLACE OLD CODE WITH THE NEW ONE HERE
#Alternatively you can use temporal aggregation (ee reducer) to create mode cloudless imagery
#Add new functionality to calculate the coverage of the composite
median_landsat, coverage = composite.get_temporal_composite(landsat_data, aoi, reducer='Median', calculate_coverage=True)
#retive thermal bands from TOA
thermal_bands, thermal_stats = optical_reflectance.get_thermal_bands(aoi, start, end, cloud_cover=40, thermal_data='L9_TOA', compute_detailed_stats=False)
median_thermal = composite.get_temporal_composite(thermal_bands, aoi, reducer='Median') #REPLACE THE OLD CODE WITH THE NEW ONE
#stacked all landsat bands and convert to float(making sure all data type are compatible)
stacked_landsat = median_landsat.addBands(median_thermal).toFloat()

# Module 2

In [ ]:
import pandas as pd
from luma_ge.classification_scheme import LULC_Scheme_Manager
manager = LULC_Scheme_Manager()

#Temporary function to display the classiifcation scheme in notebook
def display_classification_scheme(manager):
    """Display the current classification scheme in a readable format"""
    if not manager.has_classes():
        print("No classes defined yet.")
        return
    
    print("\n=== Current Classification Scheme ===")
    df = manager.get_dataframe()
    print(df.to_string(index=False))
    
    return df

In [ ]:
# Load the RESTORE+ default scheme
scheme_name = "RESTORE+ Project"
success, message = manager.load_default_scheme(scheme_name)

if success:
    print(f"✅ {message}")
else:
    print(f"❌ {message}")

# Display the loaded scheme
display_classification_scheme(manager)

classification_df = manager.get_dataframe()
LULCTable = classification_df

## Select class of interest

In [ ]:
selection = manager.store_classes_of_interest(
    scheme_name="RESTORE+ Project",
    classes_of_interest=[1, 2, 7, 14, 15]
)

print(selection)

# Module 3

In [ ]:
import pandas as pd
from luma_ge.sample_data import SyncTrainData

print(" Loading default reference training data...")
TrainEePath = 'projects/ee-rg2icraf/assets/Indonesia_lulc_Sample'
TrainField = 'kelas'

# Stopgap solution: Rename 'Land Cover Class' column to 'LULC_Type' if it exists
if 'Land Cover Class' in LULCTable.columns:
    LULCTable = LULCTable.rename(columns={'Land Cover Class': 'LULC_Type'})
    print("Column 'Land Cover Class' renamed to 'LULC_Type'")

    
try:
    print("Loading reference training data from Earth Engine...")
        
        # Load training data
    TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=LULCTable,
            aoi_geometry=aoi,
            training_ee_path=TrainEePath
        )
        
    print("Processing and validating reference data...")
        
        # Set class field
    TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)
        
        # Validate classes
    TrainDataDict = SyncTrainData.ValidClass(TrainDataDict, use_class_ids=True)
        
        # Check sufficiency
    TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)
        
        # Filter by AOI
    TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)
        
        # Create summary table
    table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
            training_data=TrainDataDict.get('training_data'),
            landcover_df=TrainDataDict.get('landcover_df'),
            class_field=TrainDataDict.get('class_field')
        )
        
    print("✅ Reference training data loaded and processed successfully!")
    print(f"Total samples: {total_samples}")
        
        # Display summary table
    display(table_df)
        
        # Store final training data
    TrainDataFinal = TrainDataDict.get('training_data')
        
        # Show validation results
    vr = TrainDataDict.get('validation_results', {})
    print(f"\nValidation Results:")
    print(f"- Total points loaded: {vr.get('total_points', 'N/A')}")
    print(f"- Points after class filter: {vr.get('points_after_class_filter', 'N/A')}")
    print(f"- Valid points (within AOI): {vr.get('valid_points', 'N/A')}")
    print(f"- Invalid classes: {len(vr.get('invalid_classes', []))}")
        
except Exception as e:
        print(f"❌ Error loading reference data: {e}")
        TrainDataFinal = None

# Module 6

For mock up purpose, default classification scheme pathway will display a prebuilt map. No classification algorithm will be conducted in this workflow.

In [ ]:
from luma_ge.classification import Generate_LULC
classifier = Generate_LULC()

## Load prebuilt map

In [ ]:
# Load the prebuilt map for the chosen default scheme

KNOWN_DEFAULT_SCHEMES = set(classifier._prebuilt_registry.keys())
chosen_scheme = "RESTORE+ Project"


if chosen_scheme in KNOWN_DEFAULT_SCHEMES:
    # Modules 3–5 skipped entirely
    print(f"Loading prebuilt map for '{chosen_scheme}' year {2018}...") # Year can be dynamic based on user input in Module 1

    result = classifier.classify_from_prebuilt(
        scheme_name    = chosen_scheme,
        aoi            = aoi,
        year           = 2018, # make this dynamic based on user input in Module 1
        scheme_classes = manager.classes,
    )

    final_map       = result["final_map"]
    present_classes = result["present_classes"]
    vis_params      = result["vis_params"]
    metrics         = result.get("accuracy_metrics")

    print(f"Loaded year    : {result['year_used']}")
    print(f"Classes in AOI : {[c['Class Name'] for c in present_classes]}") # safeguard check

    if metrics:
        print(f"Accuracy: {metrics['overall_accuracy']}")

else:
    # ── Normal RF path (Modules 3–6) ──────────────────────────────────────
    # Skip this block if a default scheme with a prebuilt map is selected
    raise NotImplementedError(
        "Custom scheme RF pipeline not yet wired up in this cell. "
        "Select a default scheme to use the prebuilt map bypass."
    )

## Reclassify according to user's selection of class of interest

In [ ]:
# Reclassify the final map according to the classes of interest

final_map, info = classifier.reclassify_map_by_classes(
    classification_map=final_map,
    classification_df=LULCTable,
    selected_classes=selection
)

## Visualization

In [ ]:
# Visualization for original map

# === Load Classification Scheme ===
scheme = LULCTable
classes = [str(x).strip() for x in scheme["LULC_Type"].tolist()]
palette = [str(x).strip() for x in scheme["Color Palette"].tolist()]
ids = scheme["ID"].tolist()
legend_dict = dict(zip(classes, palette))

# === Visualization Parameters ===
vis_params = {
    "min": min(ids),
    "max": max(ids),
    "palette": palette
}

# Prepare Reclassified Visualization
classes_of_interest = selection["classes_of_interest"]
other_class_id = 999

scheme_filtered = LULCTable[LULCTable["ID"].isin(classes_of_interest)]

reclass_names = scheme_filtered["LULC_Type"].tolist()
reclass_colors = scheme_filtered["Color Palette"].tolist()
reclass_ids = scheme_filtered["ID"].tolist()

# Add "Other"
reclass_names.append("Other")
reclass_colors.append("#BDBDBD")
reclass_ids.append(other_class_id)

# Sequential IDs for visualization
vis_ids = list(range(1, len(reclass_ids) + 1))

# Remap for visualization
vis_map = final_map.remap(reclass_ids, vis_ids)

reclass_vis_params = {
    "min": 1,
    "max": len(vis_ids),
    "palette": reclass_colors
}



Map = geemap.Map() 
Map.centerObject(aoi, 7)
# Add original map
Map.addLayer(final_map, 
             vis_params, 
             "LULC Classification")
# Add Reclassified Layer
Map.addLayer(
    vis_map,
    reclass_vis_params,
    "Reclassified LULC"
)

reclass_legend = dict(zip(reclass_names, reclass_colors))

Map.add_legend(
    title="Reclassified Land Cover",
    legend_dict=reclass_legend
)

Map

## Model Accuracy metrics

Since no model was ran for this workflow, load model accuracy statistic use a static value defined in the function

In [ ]:
print(f"Result keys: {result.keys()}")
metrics = result.get("accuracy_metrics")
print(f"Metrics content: {metrics}")
if metrics and len(metrics) > 0:
    print(f"\nAccuracy Metrics:")
    print(f"  • Overall Accuracy: {metrics.get('overall_accuracy', 'N/A')}")
    print(f"  • Kappa: {metrics.get('kappa', 'N/A')}")
    print(f"  • F1 Average: {metrics.get('average_f1_score', 'N/A')}")
    print(f"  • G-Mean Score: {metrics.get('gmean_score', 'N/A')}")
else:
    print("No accuracy metrics available")